# Notebook 04: Advanced Fine-Tuning Techniques (Bonus)

Hyperparameter tuning, DPO preference alignment, and optional scaling to larger models.

## 1. Setup

In [ ]:
import os
import json
import time
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import Dataset
from openai import OpenAI

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'

print(f'[OK] Setup complete')
print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB')

In [ ]:
# Load training data
train_samples = []
with open('../data/train_set.jsonl') as f:
    for line in f:
        train_samples.append(json.loads(line))

# Helper to load model fresh
def load_base_model(model_id=MODEL_ID):
    """Load a fresh base model in 4-bit."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return model, tokenizer

print(f'[OK] Training data: {len(train_samples)} samples')

In [ ]:
# Prepare dataset once
def prepare_dataset(samples, tokenizer):
    """Format samples and create HF Dataset."""
    def format_sample(sample):
        text = tokenizer.apply_chat_template(
            sample['messages'], tokenize=False, add_generation_prompt=False
        )
        return {'text': text}
    
    dataset = Dataset.from_list(samples)
    dataset = dataset.map(format_sample, remove_columns=dataset.column_names)
    return dataset

print('[OK] Helpers ready')

## 2. Hyperparameter Sensitivity

Let's test how different LoRA configurations affect training.

We'll run short training experiments (~50 steps each) with:
- **LoRA rank**: r=4, r=16, r=64
- **Learning rate**: 1e-5, 2e-4, 1e-3

In [ ]:
def quick_train(model, tokenizer, train_dataset, lora_r, lr, max_steps=50, output_name='test'):
    """Run a quick training experiment and return the loss history."""
    
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_r * 2,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0.05,
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    training_args = SFTConfig(
        output_dir=f'../results/hp-search/{output_name}',
        max_steps=max_steps,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        warmup_steps=5,
        logging_steps=5,
        fp16=True,
        max_seq_length=512,
        dataset_text_field='text',
        report_to='none',
    )
    
    trainer = SFTTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )
    
    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    
    losses = [(log['step'], log['loss']) for log in trainer.state.log_history if 'loss' in log]
    final_loss = losses[-1][1] if losses else float('inf')
    
    # Clean up
    del peft_model, trainer
    torch.cuda.empty_cache()
    
    return {
        'losses': losses,
        'final_loss': final_loss,
        'time': elapsed,
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad)
    }

print('[OK] Quick train function ready')

In [ ]:
# Test different LoRA ranks
rank_configs = [4, 16, 64]
rank_results = {}

for r in rank_configs:
    print(f'\nTraining with LoRA rank r={r}...')
    model, tokenizer = load_base_model()
    train_dataset = prepare_dataset(train_samples, tokenizer)
    result = quick_train(model, tokenizer, train_dataset, lora_r=r, lr=2e-4, output_name=f'rank-{r}')
    rank_results[f'r={r}'] = result
    print(f'  r={r}: final loss={result["final_loss"]:.4f}, time={result["time"]:.0f}s')
    del model
    torch.cuda.empty_cache()

print(f'\n[OK] Rank comparison complete')

In [ ]:
# Test different learning rates
lr_configs = [1e-5, 2e-4, 1e-3]
lr_results = {}

for lr in lr_configs:
    print(f'\nTraining with lr={lr}...')
    model, tokenizer = load_base_model()
    train_dataset = prepare_dataset(train_samples, tokenizer)
    result = quick_train(model, tokenizer, train_dataset, lora_r=16, lr=lr, output_name=f'lr-{lr}')
    lr_results[f'lr={lr}'] = result
    print(f'  lr={lr}: final loss={result["final_loss"]:.4f}, time={result["time"]:.0f}s')
    del model
    torch.cuda.empty_cache()

print(f'\n[OK] Learning rate comparison complete')

In [ ]:
# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Rank comparison
for name, result in rank_results.items():
    if result['losses']:
        steps, losses = zip(*result['losses'])
        ax1.plot(steps, losses, label=name, linewidth=1.5)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('LoRA Rank Comparison (lr=2e-4)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Learning rate comparison
for name, result in lr_results.items():
    if result['losses']:
        steps, losses = zip(*result['losses'])
        ax2.plot(steps, losses, label=name, linewidth=1.5)
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')
ax2.set_title('Learning Rate Comparison (r=16)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print(f'\n{"Config":<20} {"Final Loss":>12} {"Time (s)":>10}')
print('-' * 42)
for name, result in {**rank_results, **lr_results}.items():
    print(f'{name:<20} {result["final_loss"]:>12.4f} {result["time"]:>10.0f}')

## 3. DPO: Learning from Preferences

**Direct Preference Optimization (DPO)** teaches the model which responses are better by showing it pairs of:
- **Chosen**: The preferred, high-quality response
- **Rejected**: The lower-quality response we want to avoid

This is great for teaching **tone, style, and professionalism** beyond what SFT can do.

In [ ]:
# Generate preference pairs using our LLM API
# For each query: create a "chosen" (professional) and "rejected" (poor) response

preference_queries = [
    "What is your return policy?",
    "I want to return a product I bought last week.",
    "How long does shipping take?",
    "Can I change my shipping address?",
    "Do you offer international shipping?",
    "My package arrived damaged, what should I do?",
    "Can I use multiple promo codes?",
    "Do you have a loyalty program?",
    "What payment methods do you accept?",
    "I received the wrong item in my order.",
    "Can I cancel my order?",
    "Do you offer expedited shipping?",
    "What should I do if my discount code is not working?",
    "Can I order without creating an account?",
    "Do you offer gift wrapping?",
]

print(f'Generating preference pairs for {len(preference_queries)} queries...')

In [ ]:
def generate_preference_pair(query, llm_client, model_name):
    """Generate a chosen (good) and rejected (poor) response pair."""
    
    prompt = f"""For this e-commerce customer service query, generate two responses:

QUERY: {query}

1. CHOSEN (excellent): Professional, concise, helpful, uses specific policy details, friendly tone, offers next steps
2. REJECTED (poor): Vague, generic, unhelpful, too long or too short, robotic tone, doesn't address the specific question

Return ONLY a JSON object:
{{"chosen": "the excellent response", "rejected": "the poor response"}}"""
    
    response = llm_client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.8,
        max_tokens=500
    )
    
    content = response.choices[0].message.content.strip()
    if '```' in content:
        content = content.split('```')[1].replace('json', '').strip()
    
    return json.loads(content)

# Generate all pairs
preference_data = []
for i, query in enumerate(preference_queries):
    try:
        pair = generate_preference_pair(query, llm_client, LLM_MODEL)
        preference_data.append({
            'prompt': query,
            'chosen': pair['chosen'],
            'rejected': pair['rejected']
        })
        print(f'  [{i+1}/{len(preference_queries)}] Generated', end='\r')
    except Exception as e:
        print(f'  [{i+1}] Error: {str(e)[:50]}')

print(f'\n[OK] Generated {len(preference_data)} preference pairs')

In [ ]:
# Show an example
ex = preference_data[0]
print(f'Example preference pair:')
print(f'\n  QUERY:    {ex["prompt"]}')
print(f'\n  CHOSEN:   {ex["chosen"]}')
print(f'\n  REJECTED: {ex["rejected"]}')

## 4. DPO Training

In [ ]:
# Load the SFT-trained model as the starting point for DPO
model, tokenizer = load_base_model()
sft_model = PeftModel.from_pretrained(model, str(Path('../results/ecommerce-ft-v1/adapter')))
sft_model = sft_model.merge_and_unload()  # Merge SFT adapter first

print(f'[OK] SFT model loaded and merged')

In [ ]:
# Prepare DPO dataset
dpo_dataset = Dataset.from_list(preference_data)

# Apply new LoRA for DPO
dpo_lora_config = LoraConfig(
    r=8,  # Smaller rank for DPO (lighter touch)
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

sft_model = get_peft_model(sft_model, dpo_lora_config)

print(f'[OK] DPO dataset: {len(dpo_dataset)} preference pairs')
print(f'  DPO LoRA rank: 8')

In [ ]:
# DPO training
dpo_args = DPOConfig(
    output_dir='../results/ecommerce-dpo-v1',
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=10,
    fp16=True,
    max_length=512,
    max_prompt_length=256,
    beta=0.1,  # KL penalty -- controls how much DPO can change behavior
    report_to='none',
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print('Starting DPO training...')
start = time.time()
dpo_trainer.train()
elapsed = time.time() - start

print(f'\n[OK] DPO training complete in {elapsed/60:.1f} minutes')

In [ ]:
# Save DPO adapter
dpo_adapter_path = Path('../results/ecommerce-dpo-v1/adapter')
sft_model.save_pretrained(str(dpo_adapter_path))
tokenizer.save_pretrained(str(dpo_adapter_path))

print(f'[OK] DPO adapter saved to {dpo_adapter_path}')

## 4. Compare SFT vs SFT+DPO

In [ ]:
def generate_response(model, tokenizer, query, max_new_tokens=256):
    """Generate a response."""
    messages = [
        {'role': 'system', 'content': 'You are a helpful e-commerce customer service agent.'},
        {'role': 'user', 'content': query}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

# Load SFT-only model for comparison
sft_only_model, sft_tokenizer = load_base_model()
sft_only_model = PeftModel.from_pretrained(sft_only_model, str(Path('../results/ecommerce-ft-v1/adapter')))

# Compare
test_queries = [
    "What is your return policy?",
    "My package arrived damaged.",
    "Can I use multiple promo codes on one order?",
    "I received the wrong item, this is frustrating.",
    "Do you ship internationally to Japan?"
]

print('SFT vs SFT+DPO Comparison')
print('=' * 70)

for query in test_queries:
    sft_response = generate_response(sft_only_model, sft_tokenizer, query)
    dpo_response = generate_response(sft_model, tokenizer, query)
    
    print(f'\nQ: {query}')
    print(f'\n  SFT ONLY:  {sft_response[:200]}')
    print(f'\n  SFT + DPO: {dpo_response[:200]}')
    print('-' * 70)

del sft_only_model
torch.cuda.empty_cache()

## 5. Scaling to Larger Models (Optional)

Let's see how a larger model (3B) compares to our 1B model.

> **Note:** This section requires ~4-6 GB VRAM for the 3B model. Skip if your GPU doesn't have enough memory.

In [ ]:
# Clean up current models first
del sft_model
torch.cuda.empty_cache()

# Check available VRAM
free_vram = (torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / (1024**3)
print(f'Available VRAM: {free_vram:.1f} GB')

if free_vram < 4:
    print('\nNot enough VRAM for 3B model. Skipping this section.')
    print('You can try this later with more VRAM or by restarting the kernel.')
else:
    print('Enough VRAM! Loading 3B model...')

In [ ]:
# Load 3B model
LARGE_MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
# LARGE_MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'  # Fallback (3.8B, ungated)

large_model, large_tokenizer = load_base_model(LARGE_MODEL_ID)
large_dataset = prepare_dataset(train_samples, large_tokenizer)

vram_used = torch.cuda.memory_allocated() / (1024**3)
print(f'[OK] 3B model loaded')
print(f'  VRAM used: {vram_used:.1f} GB')

In [ ]:
# Quick training run on 3B model
result_3b = quick_train(
    large_model, large_tokenizer, large_dataset,
    lora_r=16, lr=2e-4, max_steps=50, output_name='3b-test'
)

print(f'\n3B model: final loss={result_3b["final_loss"]:.4f}, time={result_3b["time"]:.0f}s')

# Compare VRAM usage
print(f'\nVRAM Comparison:')
print(f'  1B model: ~2 GB (4-bit)')
print(f'  3B model: ~4 GB (4-bit)')
print(f'  7B model: ~6 GB (4-bit) [not tested]')

del large_model
torch.cuda.empty_cache()

## 6. Best Practices Checklist

Based on our experiments, here are the key takeaways:

### Data
- Data quality matters more than quantity
- Mix single-turn and multi-turn conversations
- Always keep a held-out eval set
- Use the model's native chat template

### Training
- Start with LoRA rank r=8-16, increase if underfitting
- Learning rate 2e-4 is a safe default for QLoRA
- Monitor eval loss to catch overfitting early
- 3 epochs is usually enough for small datasets

### DPO
- Apply DPO after SFT for style/tone refinement
- Use smaller LoRA rank for DPO (r=4-8)
- Use lower learning rate (5e-5)
- Beta=0.1 is a good default

### Deployment
- Merge adapters for inference speed
- Keep adapters separate during development
- Consider GGUF format for CPU/edge deployment

## 7. Summary

| Technique | What We Learned |
|-----------|----------------|
| LoRA rank | r=16 is a good default; r=64 overfits on small data |
| Learning rate | 2e-4 works well; 1e-3 can be unstable |
| DPO | Improves tone and professionalism on top of SFT |
| Model size | Larger models give better results but need more VRAM |

**Next:** Merge the adapter, build an inference pipeline, and integrate with the agentic RAG system from Part 5.